# Chapter 4: Open Financial Data and Return Engineering

This atlas follows the lecture sequence and uses deterministic financial data so
that every code module can run without an internet connection.

## Learning map

1. Open-data sources and file formats
2. Reading CSV data
3. Simple and log returns
4. Missing-value treatment
5. Daily-to-monthly and annual aggregation
6. One-sample and two-sample return tests

**Code availability:** Yes. Every code cell imports its own dependencies,
contains Chinese exam-oriented comments, and produces an interpretable result.

## How to Study This Chapter

- **First pass:** Read the concept and formula sections without running code.
- **Second pass:** Predict the output and fill in the commented key lines before execution.
- **Third pass:** Run each module independently and explain the result in one sentence.
- **Final review:** Compare related methods and identify when each method should or should not be used.

## Chapter Checklist

- Can I define every variable and state its unit?
- Can I reproduce the main formula or workflow without looking?
- Can I identify the code lines most likely to appear as blanks?
- Can I interpret the output economically or statistically?
- Can I name at least one common implementation error?


## 4.1 Open Data Workflow

Typical sources include Yahoo Finance, FRED, SEC filings, and academic factor
libraries. A robust workflow is **retrieve -> inspect -> clean -> transform ->
validate -> save**. CSV is portable and human-readable; pickle preserves Python
data types but should only be loaded from trusted sources.


## 4.2 Reading and Inspecting CSV Data

The lecture uses `pandas.read_csv`. This self-contained example creates the same table in memory and demonstrates the essential inspection steps.


### Structured Review

- **Data source:** Record where the observations come from and whether the example is online, local, or simulated.
- **Cleaning:** Inspect missing values, data types, date order, and duplicated observations before calculating returns.
- **Transformation:** Use adjacent prices for one-period returns and compounding for multi-period returns.
- **Storage:** Choose CSV for portability and pickle only for trusted Python workflows that need preserved data types.

### Formula and Workflow

1. Sort observations by date.
2. Clean the price series without inventing unsupported returns.
3. Calculate simple or log returns.
4. Aggregate with products of growth factors or sums of log returns.
5. Validate one observation manually.

### Exam Focus

- Complete `P_t / P_(t-1) - 1`, `log(P_t/P_(t-1))`, and `prod(1+R)-1`.
- Distinguish price filling from return filling.
- Explain why averaging daily returns is not the exact monthly return.

### Common Mistakes

- Calculating returns on unsorted dates.
- Keeping the first missing return in a statistical test.
- Treating a percentage value as if it were already a decimal.


In [ ]:
'''
本单元模拟从 CSV 读取股票价格，并检查字段、行数和前几条记录。
StringIO 把字符串当作文件使用，因此无需依赖外部路径。
'''
import csv
import io

csv_text = '''Date,Close,Volume
2026-01-02,100.00,1200000
2026-01-05,101.50,1350000
2026-01-06,99.80,1280000
2026-01-07,102.20,1410000
'''

# DictReader 使用首行作为列名，并将每行转换为字典
rows = list(csv.DictReader(io.StringIO(csv_text)))

# 金融计算前应把文本价格转换为浮点数
close_prices = [float(row["Close"]) for row in rows]

print("Columns =", list(rows[0]))
print("Number of observations =", len(rows))
print("First row =", rows[0])
print("Mean close =", round(sum(close_prices) / len(close_prices), 2))


**Result interpretation.** The table contains four dated observations. Explicit type conversion prevents accidental string arithmetic.

### Result Checklist

- Confirm that the sign and magnitude are economically reasonable.
- Match the output to the formula, decision rule, or statistical hypothesis described above.
- If the result differs from expectation, inspect input frequency, indexing, and parameter order first.


## 4.3 Simple Returns and Log Returns


Simple return is $R_t=P_t/P_{t-1}-1$. Log return is
$r_t=\ln(P_t/P_{t-1})=\ln(1+R_t)$. Log returns add across time,
while simple returns compound.



### Structured Review

- **Data source:** Record where the observations come from and whether the example is online, local, or simulated.
- **Cleaning:** Inspect missing values, data types, date order, and duplicated observations before calculating returns.
- **Transformation:** Use adjacent prices for one-period returns and compounding for multi-period returns.
- **Storage:** Choose CSV for portability and pickle only for trusted Python workflows that need preserved data types.

### Formula and Workflow

1. Sort observations by date.
2. Clean the price series without inventing unsupported returns.
3. Calculate simple or log returns.
4. Aggregate with products of growth factors or sums of log returns.
5. Validate one observation manually.

### Exam Focus

- Complete `P_t / P_(t-1) - 1`, `log(P_t/P_(t-1))`, and `prod(1+R)-1`.
- Distinguish price filling from return filling.
- Explain why averaging daily returns is not the exact monthly return.

### Common Mistakes

- Calculating returns on unsorted dates.
- Keeping the first missing return in a statistical test.
- Treating a percentage value as if it were already a decimal.


In [ ]:
'''
本单元根据价格序列计算百分比收益率和对数收益率。
考试填空常见位置是相邻价格的除法、减 1，以及自然对数转换。
'''
import numpy as np

prices = np.array([100.0, 101.5, 99.8, 102.2])

# 相邻价格相除后减 1，得到简单收益率
simple_returns = prices[1:] / prices[:-1] - 1

# 对价格比取自然对数，得到可加总的对数收益率
log_returns = np.log(prices[1:] / prices[:-1])

# exp(对数收益率)-1 可还原简单收益率
recovered_simple = np.exp(log_returns) - 1

print("Simple returns =", np.round(simple_returns, 6))
print("Log returns =", np.round(log_returns, 6))
print("Conversion check =", np.allclose(simple_returns, recovered_simple))


**Result interpretation.** The conversion check is true. Both definitions describe the same one-period price movement in different forms.

### Result Checklist

- Confirm that the sign and magnitude are economically reasonable.
- Match the output to the formula, decision rule, or statistical hypothesis described above.
- If the result differs from expectation, inspect input frequency, indexing, and parameter order first.


## 4.4 Missing Prices and Forward Fill


Price gaps may arise from data-source alignment or non-trading observations.
Forward fill can be reasonable for an isolated missing **price**, but missing
returns should not be filled without an economic justification.



### Structured Review

- **Data source:** Record where the observations come from and whether the example is online, local, or simulated.
- **Cleaning:** Inspect missing values, data types, date order, and duplicated observations before calculating returns.
- **Transformation:** Use adjacent prices for one-period returns and compounding for multi-period returns.
- **Storage:** Choose CSV for portability and pickle only for trusted Python workflows that need preserved data types.

### Formula and Workflow

1. Sort observations by date.
2. Clean the price series without inventing unsupported returns.
3. Calculate simple or log returns.
4. Aggregate with products of growth factors or sums of log returns.
5. Validate one observation manually.

### Exam Focus

- Complete `P_t / P_(t-1) - 1`, `log(P_t/P_(t-1))`, and `prod(1+R)-1`.
- Distinguish price filling from return filling.
- Explain why averaging daily returns is not the exact monthly return.

### Common Mistakes

- Calculating returns on unsorted dates.
- Keeping the first missing return in a statistical test.
- Treating a percentage value as if it were already a decimal.


In [ ]:
'''
本单元用 NumPy 实现价格序列的前向填充。
关键逻辑：遇到 NaN 时，用最近一个有效价格替代。
'''
import numpy as np

prices = np.array([100.0, np.nan, 101.5, np.nan, 103.0])
filled = prices.copy()

for i in range(1, len(filled)):
    # np.isnan 判断当前价格是否缺失
    if np.isnan(filled[i]):
        # 前向填充使用上一期已经确认有效的价格
        filled[i] = filled[i - 1]

returns = filled[1:] / filled[:-1] - 1

print("Original prices =", prices)
print("Forward-filled prices =", filled)
print("Returns after cleaning =", np.round(returns, 6))


**Result interpretation.** Forward-filled dates generate zero returns because the observed price is carried forward rather than economically re-estimated.

### Result Checklist

- Confirm that the sign and magnitude are economically reasonable.
- Match the output to the formula, decision rule, or statistical hypothesis described above.
- If the result differs from expectation, inspect input frequency, indexing, and parameter order first.


## 4.5 Aggregating Daily Returns


Returns must be compounded, not averaged. For a period containing daily returns
$R_d$, the period return is $\prod(1+R_d)-1$. Equivalently, sum daily log
returns and transform back.



### Structured Review

- **Data source:** Record where the observations come from and whether the example is online, local, or simulated.
- **Cleaning:** Inspect missing values, data types, date order, and duplicated observations before calculating returns.
- **Transformation:** Use adjacent prices for one-period returns and compounding for multi-period returns.
- **Storage:** Choose CSV for portability and pickle only for trusted Python workflows that need preserved data types.

### Formula and Workflow

1. Sort observations by date.
2. Clean the price series without inventing unsupported returns.
3. Calculate simple or log returns.
4. Aggregate with products of growth factors or sums of log returns.
5. Validate one observation manually.

### Exam Focus

- Complete `P_t / P_(t-1) - 1`, `log(P_t/P_(t-1))`, and `prod(1+R)-1`.
- Distinguish price filling from return filling.
- Explain why averaging daily returns is not the exact monthly return.

### Common Mistakes

- Calculating returns on unsorted dates.
- Keeping the first missing return in a statistical test.
- Treating a percentage value as if it were already a decimal.


In [ ]:
'''
本单元把带月份标签的日收益率复利为月收益率。
group label 与收益率一一对应，适合考试中的分组累计题。
'''
import numpy as np

daily_returns = np.array([0.01, -0.005, 0.004, 0.012, -0.003, 0.006])
month_labels = np.array([202601, 202601, 202601, 202602, 202602, 202602])

monthly_returns = {}
for month in np.unique(month_labels):
    # 布尔条件选出同一个月的全部日收益率
    selected = daily_returns[month_labels == month]

    # 月收益率必须对 (1+日收益率) 连乘后再减 1
    monthly_returns[int(month)] = np.prod(1 + selected) - 1

print("Monthly compounded returns =")
for month, value in monthly_returns.items():
    print(month, f"{value:.4%}")


**Result interpretation.** Each monthly result preserves compounding and can be verified using the sum of daily log returns.

### Result Checklist

- Confirm that the sign and magnitude are economically reasonable.
- Match the output to the formula, decision rule, or statistical hypothesis described above.
- If the result differs from expectation, inspect input frequency, indexing, and parameter order first.


## 4.6 Tests of Mean Returns


A one-sample t-test compares a sample mean with a target such as zero. An
independent two-sample test compares two assets. Welch's version does not assume
equal variances.



### Structured Review

- **Data source:** Record where the observations come from and whether the example is online, local, or simulated.
- **Cleaning:** Inspect missing values, data types, date order, and duplicated observations before calculating returns.
- **Transformation:** Use adjacent prices for one-period returns and compounding for multi-period returns.
- **Storage:** Choose CSV for portability and pickle only for trusted Python workflows that need preserved data types.

### Formula and Workflow

1. Sort observations by date.
2. Clean the price series without inventing unsupported returns.
3. Calculate simple or log returns.
4. Aggregate with products of growth factors or sums of log returns.
5. Validate one observation manually.

### Exam Focus

- Complete `P_t / P_(t-1) - 1`, `log(P_t/P_(t-1))`, and `prod(1+R)-1`.
- Distinguish price filling from return filling.
- Explain why averaging daily returns is not the exact monthly return.

### Common Mistakes

- Calculating returns on unsorted dates.
- Keeping the first missing return in a statistical test.
- Treating a percentage value as if it were already a decimal.


In [ ]:
'''
本单元用固定模拟收益率完成单样本和双样本 t 检验。
p 值低于显著性水平 alpha 时拒绝原假设。
'''
import numpy as np
from scipy import stats

rng = np.random.default_rng(2026)
ibm_returns = rng.normal(0.0006, 0.012, 252)
msft_returns = rng.normal(0.0009, 0.014, 252)
alpha = 0.05

# 单样本检验 H0：IBM 平均日收益率等于 0
one_sample = stats.ttest_1samp(ibm_returns, popmean=0)

# Welch t 检验 H0：两只股票平均收益率相等
two_sample = stats.ttest_ind(ibm_returns, msft_returns, equal_var=False)

print("IBM mean =", round(float(ibm_returns.mean()), 6))
print("One-sample p-value =", round(float(one_sample.pvalue), 6))
print("Two-sample p-value =", round(float(two_sample.pvalue), 6))
print("Two-sample decision =", "Reject H0" if two_sample.pvalue < alpha else "Do not reject H0")


**Result interpretation.** The conclusion is based on the p-value rather than the visual difference between sample means.

### Result Checklist

- Confirm that the sign and magnitude are economically reasonable.
- Match the output to the formula, decision rule, or statistical hypothesis described above.
- If the result differs from expectation, inspect input frequency, indexing, and parameter order first.


## 4.7 Saving, Reloading, and Dropping Missing Data


The classroom script adds a ticker column, removes incomplete observations, and
saves both CSV and pickle files. CSV is portable; pickle preserves Python
objects and should be loaded only from trusted sources.



### Structured Review

- **Data source:** Record where the observations come from and whether the example is online, local, or simulated.
- **Cleaning:** Inspect missing values, data types, date order, and duplicated observations before calculating returns.
- **Transformation:** Use adjacent prices for one-period returns and compounding for multi-period returns.
- **Storage:** Choose CSV for portability and pickle only for trusted Python workflows that need preserved data types.

### Formula and Workflow

1. Sort observations by date.
2. Clean the price series without inventing unsupported returns.
3. Calculate simple or log returns.
4. Aggregate with products of growth factors or sums of log returns.
5. Validate one observation manually.

### Exam Focus

- Complete `P_t / P_(t-1) - 1`, `log(P_t/P_(t-1))`, and `prod(1+R)-1`.
- Distinguish price filling from return filling.
- Explain why averaging daily returns is not the exact monthly return.

### Common Mistakes

- Calculating returns on unsorted dates.
- Keeping the first missing return in a statistical test.
- Treating a percentage value as if it were already a decimal.


In [ ]:
'''
本单元补充源代码中的 ticker、dropna、CSV 和 pickle 保存流程。
使用临时目录，运行后自动清理，不修改用户工作目录。
'''
import csv
import math
import pickle
import tempfile
from pathlib import Path

records = [
    {"Date": "2026-01-02", "Close": 100.0, "Return": None, "Ticker": "IBM"},
    {"Date": "2026-01-05", "Close": 101.5, "Return": 0.015, "Ticker": "IBM"},
    {"Date": "2026-01-06", "Close": 99.8, "Return": -0.016749, "Ticker": "IBM"},
]

# dropna 的核心逻辑：只保留 Return 不是缺失值的记录
clean_records = [row for row in records if row["Return"] is not None]

with tempfile.TemporaryDirectory() as folder:
    csv_path = Path(folder) / "ibm_daily.csv"
    pickle_path = Path(folder) / "ibm_daily.pkl"

    # CSV 写入需要先指定列名 fieldnames
    with csv_path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=records[0].keys())
        writer.writeheader()
        writer.writerows(clean_records)

    # pickle 直接序列化 Python 对象并保留数据类型
    with pickle_path.open("wb") as file:
        pickle.dump(clean_records, file)

    with pickle_path.open("rb") as file:
        reloaded = pickle.load(file)

    print("Clean row count =", len(clean_records))
    print("CSV created =", csv_path.exists())
    print("Reloaded pickle =", reloaded)


**Result interpretation.** The missing first return is removed, and the cleaned records can be restored from pickle with their numeric types intact.

### Result Checklist

- Confirm that the sign and magnitude are economically reasonable.
- Match the output to the formula, decision rule, or statistical hypothesis described above.
- If the result differs from expectation, inspect input frequency, indexing, and parameter order first.
